In [3]:
import pandas as pd

materials = pd.read_csv('../../data/processed/materials.csv')
products = pd.read_csv('../../data/processed/products.csv')

print(materials.isnull().sum())
print(products.isnull().sum())

material_id                                   0
packaging_type                                0
material_type                                 0
suitable_product_categories                   0
recommended_packaging_use_cases               0
supplier_region                               0
recyclability_percent                         0
recyclability_category                        0
recycled_content_percent                      0
reusability_percent                           0
biodegradation_time_days                      0
end_of_life_disposal_percent                  0
carbon_footprint_kg_co2_per_unit              0
co2_emission_per_kg                           0
waste_reduction_impact_percent                0
sustainability_target_progress_percent        0
load_handling_score                           0
moisture_resistance_score                     0
thermal_resistance_score                      0
cost_per_unit_usd                             0
annual_usage_units                      

## Missing Value Check

Both `materials.csv` and `products.csv` (processed) show zero missing 
values. This is because the single incomplete row found in the raw 
materials data (MAT_0404, missing 14 of 23 values) was already removed 
during the ingestion/cleaning step, since too much of that row was 
missing to reliably impute.

No further imputation is needed on the current dataset. However, 
imputers are still set up below as defensive code, in case future 
data batches contain gaps.

In [4]:
from sklearn.impute import SimpleImputer

# Numeric columns (median is robust to outliers like biodegradation_time_days)
num_imputer = SimpleImputer(strategy="median")

# Categorical columns
cat_imputer = SimpleImputer(strategy="most_frequent")

In [5]:
numeric_cols = materials.select_dtypes(include='number').columns
categorical_cols = materials.select_dtypes(include='object').columns

num_imputer.fit(materials[numeric_cols])
cat_imputer.fit(materials[categorical_cols])

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [6]:
materials[numeric_cols] = num_imputer.transform(materials[numeric_cols])
materials[categorical_cols] = cat_imputer.transform(materials[categorical_cols])

In [7]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
import joblib
import os

# ---- 1. Nominal columns -> One-Hot Encoding ----
nominal_cols = ['material_type', 'packaging_type', 'supplier_region']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_encoded = ohe.fit_transform(materials[nominal_cols])
ohe_df = pd.DataFrame(ohe_encoded, columns=ohe.get_feature_names_out(nominal_cols))

# ---- 2. Ordinal column -> Label Encoding (manual order, Low < Medium < High) ----
order_map = {'Low': 0, 'Medium': 1, 'High': 2}
materials['recyclability_category_encoded'] = materials['recyclability_category'].map(order_map)

# ---- 3. Numeric columns -> scale to 0-1 range ----
numeric_feature_cols = [
    'recyclability_percent', 'recycled_content_percent', 'reusability_percent',
    'biodegradation_time_days', 'end_of_life_disposal_percent',
    'carbon_footprint_kg_co2_per_unit', 'co2_emission_per_kg',
    'waste_reduction_impact_percent', 'sustainability_target_progress_percent',
    'load_handling_score', 'moisture_resistance_score', 'thermal_resistance_score',
    'cost_per_unit_usd', 'annual_usage_units', 'total_material_weight_tons',
    'supplier_sustainability_compliance_percent'
]
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(materials[numeric_feature_cols])
scaled_df = pd.DataFrame(scaled_values, columns=[f"{c}_scaled" for c in numeric_feature_cols])

# ---- 4. Combine everything into the final ML-ready dataframe ----
materials_encoded = pd.concat(
    [materials[['material_id', 'recyclability_category_encoded']].reset_index(drop=True),
     ohe_df.reset_index(drop=True),
     scaled_df.reset_index(drop=True)],
    axis=1
)

# ---- 5. Save encoders/scaler for reuse later ----
os.makedirs('../../models/encoders', exist_ok=True)
os.makedirs('../../models/scalers', exist_ok=True)
joblib.dump(ohe, '../../models/encoders/ohe_encoder.pkl')
joblib.dump(scaler, '../../models/scalers/numeric_scaler.pkl')

# ---- 6. Export final encoded dataset ----
os.makedirs('../../data/model_ready', exist_ok=True)
materials_encoded.to_csv('../../data/model_ready/materials_final_encoded.csv', index=False)

print("Shape:", materials_encoded.shape)
materials_encoded.head()

Shape: (403, 34)


,material_id,recyclability_category_encoded,material_type_Cardboard,material_type_Paper/Bio-Based,material_type_Plastic,material_type_Steel,packaging_type_Bubble Wrap (Minimal Use),packaging_type_Cardboard Boxes,packaging_type_Foldable & Stackable Containers,packaging_type_Plastic Totes & Pallets,...,co2_emission_per_kg_scaled,waste_reduction_impact_percent_scaled,sustainability_target_progress_percent_scaled,load_handling_score_scaled,moisture_resistance_score_scaled,thermal_resistance_score_scaled,cost_per_unit_usd_scaled,annual_usage_units_scaled,total_material_weight_tons_scaled,supplier_sustainability_compliance_percent_scaled
0,MAT_0001,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.087542,0.569231,0.739130,0.555556,0.444444,0.250,0.078866,0.963447,0.130382,0.594595
1,MAT_0002,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.013468,0.676923,0.956522,0.222222,0.222222,0.125,0.062549,0.659777,0.080857,0.675676
2,MAT_0003,2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.000000,0.938462,0.543478,0.777778,0.888889,0.875,0.963092,0.000000,0.977158,0.675676
3,MAT_0004,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.010101,0.692308,0.717391,0.000000,0.000000,0.250,0.086247,0.671735,0.103295,0.486486
4,MAT_0005,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.010101,0.569231,0.760870,0.222222,0.111111,0.125,0.057498,0.606917,0.081666,0.486486


In [8]:
materials['recyclability_category_encoded'].value_counts()

recyclability_category_encoded
2    205
1    198
Name: count, dtype: int64

In [9]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import joblib
import os

# ---- 1. Nominal columns -> One-Hot Encoding ----
nominal_cols = ['category', 'shipping_type']
ohe_products = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_encoded = ohe_products.fit_transform(products[nominal_cols])
ohe_df = pd.DataFrame(ohe_encoded, columns=ohe_products.get_feature_names_out(nominal_cols))

# ---- 2. Numeric columns -> scale to 0-1 range ----
numeric_feature_cols = ['product_weight_kg', 'fragility_index']
scaler_products = MinMaxScaler()
scaled_values = scaler_products.fit_transform(products[numeric_feature_cols])
scaled_df = pd.DataFrame(scaled_values, columns=[f"{c}_scaled" for c in numeric_feature_cols])

# ---- 3. Combine into final ML-ready dataframe ----
products_encoded = pd.concat(
    [products[['product_id']].reset_index(drop=True),
     ohe_df.reset_index(drop=True),
     scaled_df.reset_index(drop=True)],
    axis=1
)

# ---- 4. Save encoders/scaler ----
joblib.dump(ohe_products, '../../models/encoders/ohe_encoder_products.pkl')
joblib.dump(scaler_products, '../../models/scalers/numeric_scaler_products.pkl')

# ---- 5. Export ----
products_encoded.to_csv('../../data/model_ready/products_final_encoded.csv', index=False)

print("Shape:", products_encoded.shape)
products_encoded.head()

Shape: (1500, 12)


,product_id,category_Cosmetics,category_Drinkware,category_Electronics,category_Food,category_Paper Product,category_Pharmacy,shipping_type_Air,shipping_type_Road,shipping_type_Sea,product_weight_kg_scaled,fragility_index_scaled
0,1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.030749,0.25
1,2,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.017380,1.00
2,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.010695,0.75
3,4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.004011,0.50
4,5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.044118,0.25


In [10]:
materials_encoded = pd.read_csv('../../data/model_ready/materials_final_encoded.csv')
materials_encoded.head()

,material_id,recyclability_category_encoded,material_type_Cardboard,material_type_Paper/Bio-Based,material_type_Plastic,material_type_Steel,packaging_type_Bubble Wrap (Minimal Use),packaging_type_Cardboard Boxes,packaging_type_Foldable & Stackable Containers,packaging_type_Plastic Totes & Pallets,...,co2_emission_per_kg_scaled,waste_reduction_impact_percent_scaled,sustainability_target_progress_percent_scaled,load_handling_score_scaled,moisture_resistance_score_scaled,thermal_resistance_score_scaled,cost_per_unit_usd_scaled,annual_usage_units_scaled,total_material_weight_tons_scaled,supplier_sustainability_compliance_percent_scaled
0,MAT_0001,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.087542,0.569231,0.739130,0.555556,0.444444,0.250,0.078866,0.963447,0.130382,0.594595
1,MAT_0002,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.013468,0.676923,0.956522,0.222222,0.222222,0.125,0.062549,0.659777,0.080857,0.675676
2,MAT_0003,2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.000000,0.938462,0.543478,0.777778,0.888889,0.875,0.963092,0.000000,0.977158,0.675676
3,MAT_0004,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.010101,0.692308,0.717391,0.000000,0.000000,0.250,0.086247,0.671735,0.103295,0.486486
4,MAT_0005,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.010101,0.569231,0.760870,0.222222,0.111111,0.125,0.057498,0.606917,0.081666,0.486486


In [11]:
materials_encoded['co2_score'] = 1 - materials_encoded['co2_emission_per_kg_scaled']
materials_encoded['green_score'] = 1 - materials_encoded['biodegradation_time_days_scaled']
materials_encoded['recyclability_score'] = materials_encoded['recyclability_category_encoded'] / 2
materials_encoded['durability_score'] = materials_encoded[
    ['load_handling_score_scaled', 'moisture_resistance_score_scaled', 'thermal_resistance_score_scaled']
].mean(axis=1)
materials_encoded['cost_score'] = 1 - materials_encoded['cost_per_unit_usd_scaled']

materials_encoded[['co2_score', 'green_score', 'recyclability_score', 'durability_score', 'cost_score']].head()

,co2_score,green_score,recyclability_score,durability_score,cost_score
0,0.912458,0.999129,1.0,0.416667,0.921134
1,0.986532,0.999811,1.0,0.189815,0.937451
2,0.000000,0.000000,1.0,0.847222,0.036908
3,0.989899,0.999811,1.0,0.083333,0.913753
4,0.989899,0.999495,1.0,0.152778,0.942502


In [12]:
materials_encoded['co2_impact_index'] = (
    0.5 * materials_encoded['co2_score'] +
    0.3 * materials_encoded['green_score'] +
    0.2 * materials_encoded['recyclability_score']
) * 100

materials_encoded['cost_efficiency_index'] = (
    0.5 * materials_encoded['cost_score'] +
    0.3 * materials_encoded['durability_score'] +
    0.2 * materials_encoded['recyclability_score']
) * 100

materials_encoded['material_suitability_score'] = materials_encoded[
    ['load_handling_score_scaled', 'moisture_resistance_score_scaled', 'thermal_resistance_score_scaled']
].mean(axis=1) * 100

materials_encoded[['material_id', 'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']].head()

,material_id,co2_impact_index,cost_efficiency_index,material_suitability_score
0,MAT_0001,95.596755,78.556721,41.666667
1,MAT_0002,99.320938,72.567016,18.981481
2,MAT_0003,20.000000,47.262044,84.722222
3,MAT_0004,99.489288,68.187646,8.333333
4,MAT_0005,99.479798,71.708430,15.277778


In [13]:
materials_encoded.to_csv('../../data/model_ready/materials_with_features.csv', index=False)
print("Saved:", materials_encoded.shape)

Saved: (403, 42)


In [14]:
materials_features = pd.read_csv('../../data/model_ready/materials_with_features.csv')
materials_clean = pd.read_csv('../../data/processed/materials.csv')

# Merge to get the raw, real-world target values
data = materials_features.merge(
    materials_clean[['material_id', 'cost_per_unit_usd', 'co2_emission_per_kg']],
    on='material_id'
)
data.shape

(403, 44)